# 01 — Exploratory Data Analysis

Explore the two core datasets used for next-day return prediction:
- **CRSP daily** — stock returns (target + momentum / volatility / size features)
- **Daily futures** — macro signals (lagged returns as features)

Key questions:
1. What is the date coverage and stock universe?
2. What does the return distribution look like (tails, skew)?
3. Which futures instruments have sufficient history?
4. Are there data quality issues (nulls, outliers, stale prices)?
5. What does a prototype of our features (momentum, vol) look like?

In [3]:
import sys, os
sys.path.insert(0, os.path.join('..'))   # project root on path

import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import wrds
import polars as pl
import pyarrow
import config

from src.data_loading import load_crsp, load_futures, load_crsp_polars, wrds_fetch
from src.preprocessing import clean_crsp, clean_futures

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.float_format', '{:.4f}'.format)
print('Config window:', config.START_DATE, '→', config.END_DATE)

Config window: 2000-01-01 → 2020-11-30


## 1  CRSP Daily Returns

In [4]:
# Function drops non complete PERMNO
df = load_crsp_polars()

permno_list = df["PERMNO"].drop_nulls().unique().to_list()
print(f'PERMNO remaining : {len(permno_list)}')

df.to_pandas().to_parquet(config.CRSP_PATH_CLEAN)

PERMNO remaining : 7355


In [3]:
wrds_df = wrds_fetch(permno_list)

/!| PLEASE FILL CREDENTIALS /!|
WRDS recommends setting up a .pgpass file.
pgpass file created at C:\Users\dario\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [ ]:
wrds_df = wrds_df.with_columns(
    pl.col('date').cast(pl.Date)
)

df = df.join(wrds_df, on=['PERMNO', 'date'], how='left')
df =df.with_columns(
    (pl.col('DlyPrc').abs() * pl.col('ShrOut') * 1000).alias('MktCap'),
    ((pl.col('DlyPrc').abs() * pl.col('ShrOut') * 1000).clip(lower_bound=1).log()).alias('log_MktCap')
)

In [21]:
# Loading the full file takes ~30 s
crsp_raw = load_crsp()
print(f'Raw shape : {crsp_raw.shape}')
print(f'Columns   : {list(crsp_raw.columns)}')
crsp_raw.head(3)

WRDS recommends setting up a .pgpass file.
pgpass file created at C:\Users\dario\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Raw shape : (47333141, 15)
Columns   : ['PERMNO', 'HdrCUSIP', 'CUSIP', 'Ticker', 'TradingSymbol', 'PERMCO', 'SICCD', 'NAICS', 'date', 'ret', 'mkt_ret', 'DlyPrc', 'ShrOut', 'mktcap', 'log_mktcap']


,PERMNO,HdrCUSIP,CUSIP,Ticker,TradingSymbol,PERMCO,SICCD,NAICS,date,ret,mkt_ret,DlyPrc,ShrOut,mktcap,log_mktcap
0,10001,36720410,29274A10,EWST,EWST,7953,4920,0,2000-01-03,0.0074,-0.0095,8.5625,2450.0000,20978125.0000,16.8590
1,10001,36720410,29274A10,EWST,EWST,7953,4920,0,2000-01-04,-0.0146,-0.0383,8.4375,2450.0000,20671875.0000,16.8443
2,10001,36720410,29274A10,EWST,EWST,7953,4920,0,2000-01-05,0.0148,0.0019,8.5625,2450.0000,20978125.0000,16.8590


In [ ]:
'''
db = wrds.Connection(wrds_username="leowunderli")
# Fetch price and shares data for market cap calculations.
permno_list = ",".join(str(int(p)) for p in crsp_raw["PERMNO"].dropna().unique())
query = f"""
    SELECT permno, date, prc AS dlyprc, shrout
    FROM crsp.dsf
    WHERE date BETWEEN '{config.START_DATE}' AND '{config.END_DATE}'
      AND permno IN ({permno_list})
    """
wrds_df = db.raw_sql(query, date_cols=["date"])
wrds_df.rename(columns={"permno": "PERMNO", "dlyprc": "DlyPrc", "shrout": "ShrOut"}, inplace=True)

df = crsp_raw.merge(wrds_df, on=["PERMNO", "date"], how="left")
df["mktcap"] = df["DlyPrc"].abs() * df["ShrOut"] * 1_000
df["log_mktcap"] = np.log(df["mktcap"].clip(lower=1))

'''

In [27]:
print('=== Raw CRSP ===')
print(f'Date range     : {crsp_raw["date"].min().date()} → {crsp_raw["date"].max().date()}')
print(f'Unique PERMNOs : {crsp_raw["PERMNO"].nunique():,}')
print(f'Total obs      : {len(crsp_raw):,}')

null_ret = crsp_raw['ret'].isna().sum()
print(f'\nMissing ret    : {null_ret:,}  ({null_ret/len(crsp_raw)*100:.2f}%)')

# Check if price / shares are present (needed for market cap)
mktcap_cols = {'mktcap', 'log_mktcap'} & set(crsp_raw.columns)
print(f'\nMarket cap cols: {mktcap_cols or "NONE — re-download CRSP with DlyPrc + ShrOut from WRDS"}')

=== Raw CRSP ===
Date range     : 2000-01-03 → 2024-12-31
Unique PERMNOs : 23,660
Total obs      : 47,333,141

Missing ret    : 633,536  (1.34%)

Market cap cols: {'mktcap', 'log_mktcap'}


In [28]:
crsp = clean_crsp(crsp_raw)
dropped = len(crsp_raw) - len(crsp)
print(f'After cleaning : {len(crsp):,} obs, {crsp["PERMNO"].nunique():,} stocks')
print(f'Date range     : {crsp["date"].min().date()} → {crsp["date"].max().date()}')
print(f'Dropped        : {dropped:,} rows ({dropped/len(crsp_raw)*100:.1f}%)')


After cleaning : 37,206,893 obs, 19,075 stocks
Date range     : 2000-01-03 → 2020-11-30
Dropped        : 10,126,248 rows (21.4%)


### 1.1  Return distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram clipped for display
ret_clipped = crsp['ret'].clip(-0.20, 0.20)
axes[0].hist(ret_clipped, bins=200, color='steelblue', edgecolor='none', alpha=0.8)
axes[0].set_title('Daily return distribution (clipped ±20%)')
axes[0].set_xlabel('Return')
axes[0].set_ylabel('Count')

# Tail percentiles
pcts = [0.1, 0.5, 1, 5, 10, 25, 50, 75, 90, 95, 99, 99.5, 99.9]
quantiles = np.percentile(crsp['ret'].dropna(), pcts)
axes[1].barh(range(len(pcts)), quantiles,
             color=['tomato' if q < 0 else 'steelblue' for q in quantiles])
axes[1].set_yticks(range(len(pcts)))
axes[1].set_yticklabels([f'p{p}' for p in pcts])
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set_title('Return percentiles')

# Cross-sectional distribution of median returns per stock
med_by_stock = crsp.groupby('PERMNO')['ret'].median()
axes[2].hist(med_by_stock.clip(-0.005, 0.005), bins=100, color='darkorange', alpha=0.8)
axes[2].set_title('Median daily return per stock')
axes[2].set_xlabel('Median return')

plt.tight_layout()
os.makedirs(str(config.PLOTS_DIR), exist_ok=True)
plt.savefig(f'{config.PLOTS_DIR}/crsp_return_distribution.png', bbox_inches='tight')
plt.show()

print(crsp['ret'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))

### 1.2  Universe size over time

In [ ]:
monthly_counts = (
    crsp.set_index('date')
    .resample('ME')['PERMNO']
    .nunique()
)

fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(monthly_counts.index, monthly_counts.values, alpha=0.4, color='steelblue')
ax.plot(monthly_counts.index, monthly_counts.values, color='steelblue', lw=1)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_title('Number of active stocks per month')
ax.set_ylabel('# stocks')
plt.tight_layout()
plt.savefig(f'{config.PLOTS_DIR}/crsp_universe_size.png', bbox_inches='tight')
plt.show()

print(f'Min: {monthly_counts.min():,}   Max: {monthly_counts.max():,}   Median: {int(monthly_counts.median()):,}')

### 1.3  Cumulative market return vs equal-weighted average

In [ ]:
daily_agg = crsp.groupby('date').agg(
    ew_ret=('ret', 'mean'),
    mkt_ret=('mkt_ret', 'first'),
    n_stocks=('PERMNO', 'nunique'),
).reset_index()

daily_agg['cum_ew']  = np.log1p(daily_agg['ew_ret']).cumsum()
daily_agg['cum_mkt'] = np.log1p(daily_agg['mkt_ret']).cumsum()

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

axes[0].plot(daily_agg['date'], daily_agg['cum_mkt'], label='S&P 500 (sprtrn)', color='black', lw=1)
axes[0].plot(daily_agg['date'], daily_agg['cum_ew'],  label='EW avg stock',     color='steelblue', lw=1, alpha=0.8)
axes[0].set_title('Cumulative log return')
axes[0].legend()
axes[0].set_ylabel('Cum. log ret')

axes[1].plot(daily_agg['date'], daily_agg['n_stocks'], color='darkorange', lw=0.8)
axes[1].set_title('Daily stock count')
axes[1].set_ylabel('# stocks')

plt.tight_layout()
plt.savefig(f'{config.PLOTS_DIR}/crsp_cumulative_returns.png', bbox_inches='tight')
plt.show()

### 1.4  Observations per stock (coverage)

In [ ]:
obs_per_stock = crsp.groupby('PERMNO').size()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(obs_per_stock, bins=100, color='steelblue', alpha=0.8)
ax.axvline(252, color='red', lw=1.5, ls='--', label='252 days (1 year)')
ax.axvline(obs_per_stock.median(), color='black', lw=1.5, ls='--',
           label=f'Median = {int(obs_per_stock.median())}')
ax.set_title('Trading days per stock in dataset')
ax.set_xlabel('# trading days')
ax.legend()
plt.tight_layout()
plt.show()

print(obs_per_stock.describe(percentiles=[.1, .25, .5, .75, .9]))
print(f'\nStocks with <252 days (< 1y): {(obs_per_stock < 252).sum():,} ({(obs_per_stock < 252).mean()*100:.1f}%)')

### 1.5  Market cap — WRDS download required

> **Action**: Re-download CRSP from WRDS and add `DlyPrc` (closing price) and `ShrOut` (shares outstanding in thousands).  
> `preprocessing.clean_crsp()` will then compute `log_mktcap = log(|DlyPrc| × ShrOut × 1000)` automatically.

In [ ]:
if 'log_mktcap' in crsp.columns:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    crsp.groupby('date')['log_mktcap'].median().plot(ax=axes[0], color='steelblue')
    axes[0].set_title('Median log market cap over time')
    axes[1].hist(crsp['log_mktcap'].dropna(), bins=100, color='darkorange', alpha=0.8)
    axes[1].set_title('Distribution of log market cap')
    plt.tight_layout(); plt.show()
    print(crsp['log_mktcap'].describe())
else:
    print('log_mktcap not yet available — add DlyPrc + ShrOut to the CRSP download.')

---
## 3  Feature Prototype

Compute momentum and volatility on a small subset to verify the logic before running on the full 47M-row panel.

In [ ]:
from src.feature_engineering import add_momentum, add_reversal, add_volatility, add_target

# 20 stocks with the longest history in the cleaned dataset
top_stocks = crsp.groupby('PERMNO').size().nlargest(20).index
sample = crsp[crsp['PERMNO'].isin(top_stocks)].copy()

sample = add_momentum(sample)
sample = add_reversal(sample)
sample = add_volatility(sample)
sample = add_target(sample)

mom_cols = [f'mom_{w}d' for w in config.MOMENTUM_WINDOWS]
vol_cols = [f'vol_{w}d' for w in config.VOLATILITY_WINDOWS]

print('Feature NaN rates (should only be non-zero for the warm-up period):')
for c in ['reversal_1d'] + mom_cols + vol_cols + ['target']:
    print(f'  {c:20s}: {sample[c].isna().mean():.3f}')

In [ ]:
# Visualise features for one stock
permno = top_stocks[0]
stock  = sample[sample['PERMNO'] == permno].set_index('date')

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

axes[0].plot(stock.index, np.log1p(stock['ret']).cumsum(), color='black', lw=1)
axes[0].set_title(f'Cumulative log return — PERMNO {permno}')
axes[0].set_ylabel('Cum. log ret')

for col, lw, alpha in zip(mom_cols, [2, 2, 1.5, 1.5, 1], [1, 0.8, 0.7, 0.6, 0.5]):
    axes[1].plot(stock.index, stock[col], label=col, lw=lw, alpha=alpha)
axes[1].axhline(0, color='black', lw=0.5)
axes[1].set_title('Momentum features')
axes[1].set_ylabel('Cum. log ret')
axes[1].legend(ncol=3, fontsize=8)

for col, lw in zip(vol_cols, [2, 1.5, 1]):
    axes[2].plot(stock.index, stock[col], label=col, lw=lw)
axes[2].set_title('Volatility features (annualised)')
axes[2].set_ylabel('Ann. vol')
axes[2].legend(ncol=3, fontsize=8)

plt.tight_layout()
plt.savefig(f'{config.PLOTS_DIR}/feature_prototype_single_stock.png', bbox_inches='tight')
plt.show()

In [ ]:
# Rank IC: spearman correlation of each feature with next-day return
feature_cols = ['reversal_1d'] + mom_cols + vol_cols

ic_by_date = (
    sample.dropna(subset=feature_cols + ['target'])
    .groupby('date')
    .apply(lambda g: g[feature_cols + ['target']]
           .corr(method='spearman')['target']
           .drop('target'),
           include_groups=False)
)

mean_ic = ic_by_date.mean()
t_stat  = ic_by_date.mean() / (ic_by_date.std() / np.sqrt(len(ic_by_date)))

ic_summary = pd.DataFrame({'Mean IC': mean_ic, 't-stat': t_stat})
print('Rank IC (feature vs next-day return) — 20-stock sample:')
print(ic_summary.to_string())

---
## 4  Dataset Alignment Summary

In [ ]:
crsp_window = crsp[(crsp['date'] >= config.START_DATE) & (crsp['date'] <= config.END_DATE)]
fut_window  = futures[(futures['date'] >= config.START_DATE) & (futures['date'] <= config.END_DATE)]

n_stock_features   = 1 + len(config.MOMENTUM_WINDOWS) + len(config.VOLATILITY_WINDOWS)
n_futures_features = len(fut_cols) * len(config.FUTURES_LAGS)
total_features     = n_stock_features + 1 + n_futures_features   # +1 for size (pending)

print('=== Dataset alignment in project window ===')
print(f'CRSP rows             : {len(crsp_window):>12,}')
print(f'CRSP unique stocks    : {crsp_window["PERMNO"].nunique():>12,}')
print(f'CRSP trading days     : {crsp_window["date"].nunique():>12,}')
print()
print(f'Futures rows          : {len(fut_window):>12,}')
print(f'Futures instruments   : {len(fut_cols):>12,}')
print(f'Futures trading days  : {fut_window["date"].nunique():>12,}')
print()
print(f'Expected feature count: {total_features}')
print(f'  reversal            :  1')
print(f'  momentum            :  {len(config.MOMENTUM_WINDOWS)}  ({config.MOMENTUM_WINDOWS})')
print(f'  volatility          :  {len(config.VOLATILITY_WINDOWS)}  ({config.VOLATILITY_WINDOWS})')
print(f'  size (log mktcap)   :  1  (pending WRDS download)')
print(f'  futures macro       :  {n_futures_features}  ({len(fut_cols)} instruments × {len(config.FUTURES_LAGS)} lags)')

---
## 5  Action items before `02_features.ipynb`

| # | Task | Status |
|---|------|--------|
| 2 | Decide minimum stock history threshold (e.g. ≥252 days) | ⬜ |
| 3 | Decide whether to exclude micro-caps (size filter) | ⬜ |
| 4 | Confirm train / val / test split (`config.py`: 2000–2015 / 2016–2018 / 2019–2020) | ⬜ |